In [1]:
import pandas as pd
import numpy as np
print("Kernel works")

Kernel works


In [2]:
df = pd.read_csv('112_final.csv', encoding='utf-8')
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Rows: 1,970,922
Columns: ['incident_type_code', 'incident_id', 'year', 'month', 'higher_level_incident_type', 'lower_level_incident_type', 'object_id', 'longitude', 'latitude', 'data_quality_flag']


In [3]:
EXCLUDE_TYPES = {
    'Atsisakė/Atsakytas', 'Konsultacija', 'GMP konsultacija',
    'Perduota kitoms tarnyboms', 'Skambutis ne GMP tikslais',
    'Pagalba 112', 'BPC TRUKDANTIS', 'Testavimo pratybos',
    'PGT pratybos', 'Pratybos (SA)', 'Dubliuotas kvietimas',
    'Neklasifikuoti', 'PGT neklasifikuoti', 'PGT neklasifikuoti (SA)',
    'Skambinančiajam reikalaujant', 'Pervežimas', 'Skubus pervežimas',
    'Komercinis pervežimas', 'Budėjimas renginyje, laidotuvėse',
    'SPT', 'Valstybės tarnautojų korupcija',
    'POLICIJOS pareigūnų korupcija *', 'Neigiama informacija internete',
    'Pagalba kitoms tarnyboms', 'Pagalba specialiosioms tarnyboms',
    'Pagalbos tarnybų pareigūnų prašymas', '116000',
}

df_clean = df[
    (df['data_quality_flag'] == 'ok') &
    (~df['lower_level_incident_type'].isin(EXCLUDE_TYPES))
].copy()

print(f"Original rows:       {len(df):,}")
print(f"After filtering:     {len(df_clean):,}")
print(f"Unique target types: {df_clean['lower_level_incident_type'].nunique()}")
print(f"\nMost common target types:")
print(df_clean['lower_level_incident_type'].value_counts().head(10).to_string())

Original rows:       1,970,922
After filtering:     1,692,116
Unique target types: 127

Most common target types:
lower_level_incident_type
GMP įvykis                              695756
KET pažeidimas                          175987
Įvairūs viešosios tvarkos pažeidimai    151815
Turtinė veika anksčiau                   81887
Smurtas artimoje aplinkoje               77728
Kompleksinis įvykis                      68024
Pavojus eismo saugumui                   41543
Nusikaltimai asmeniui dabar              31649
Turtinė veika dabar                      28617
Lavonas                                  20117


In [4]:
features = df_clean[['latitude', 'longitude', 'month', 'year']]
target = df_clean['lower_level_incident_type']

print(f"Features shape: {features.shape}")
print(f"Target shape:   {target.shape}")

Features shape: (1692116, 4)
Target shape:   (1692116,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, target,
    test_size=0.2,
    random_state=42,
    stratify=target if target.value_counts().min() >= 2 else None,
)

print(f"Train: {len(X_train):,} rows")
print(f"Test:  {len(X_test):,} rows")
print(f"Train classes: {y_train.nunique()}")
print(f"Test classes:  {y_test.nunique()}")

Train: 1,353,692 rows
Test:  338,424 rows
Train classes: 124
Test classes:  121


In [8]:
class_counts = df_clean['lower_level_incident_type'].value_counts()
print(f"Total rare classes (< 10 rows): {(class_counts < 10).sum()}")
print(f"Rare classes (< 100 rows): {(class_counts < 100).sum()}")

Total rare classes (< 10 rows): 15
Rare classes (< 100 rows): 37


In [9]:
MIN_EXAMPLES_PER_CLASS = 10
keep_classes = class_counts[class_counts >= MIN_EXAMPLES_PER_CLASS].index
df_clean = df_clean[df_clean['lower_level_incident_type'].isin(keep_classes)].copy()

print(f"Rows after dropping rare classes: {len(df_clean):,}")
print(f"Unique target classes:            {df_clean['lower_level_incident_type'].nunique()}")

Rows after dropping rare classes: 1,692,053
Unique target classes:            112


In [12]:
print(f"df_clean rows: {len(df_clean):,}")
print(f"df_clean unique classes: {df_clean['lower_level_incident_type'].nunique()}")
print(f"y_train rows: {len(y_train):,}")
print(f"y_train unique classes: {y_train.nunique()}")
print(f"y_train min class size: {y_train.value_counts().min()}")

df_clean rows: 1,692,053
df_clean unique classes: 112
y_train rows: 1,353,692
y_train unique classes: 124
y_train min class size: 1


In [13]:
features = df_clean[['latitude', 'longitude', 'month', 'year']]
target = df_clean['lower_level_incident_type']

X_train, X_test, y_train, y_test = train_test_split(
    features, target,
    test_size=0.2,
    random_state=42,
    stratify=target if target.value_counts().min() >= 2 else None,
)

print(f"Train: {len(X_train):,} rows")
print(f"Test:  {len(X_test):,} rows")
print(f"Train classes: {y_train.nunique()}")
print(f"Test classes:  {y_test.nunique()}")
print(f"Min class size in train: {y_train.value_counts().min()}")

Train: 1,353,642 rows
Test:  338,411 rows
Train classes: 112
Test classes:  112
Min class size in train: 9


In [14]:
from sklearn.ensemble import HistGradientBoostingClassifier
import time

print("Training HistGradientBoostingClassifier...")
t = time.time()

model = HistGradientBoostingClassifier(
    max_iter=100,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=1,
)
model.fit(X_train, y_train)

print(f"\nTrained in {(time.time() - t) / 60:.1f} minutes")

Training HistGradientBoostingClassifier...
Binning 0.039 GB of training data: 0.155 s
Binning 0.004 GB of validation data: 0.004 s
Fitting gradient boosted rounds:
Fit 1120 trees in 83.502 s, (34691 total leaves)
Time spent computing histograms: 23.813s
Time spent finding best splits:  1.653s
Time spent applying splits:      13.797s
Time spent predicting:           8.793s

Trained in 1.4 minutes


In [15]:
import numpy as np
from sklearn.metrics import accuracy_score

# Top-1: model's single best prediction
y_pred = model.predict(X_test)
top1_acc = accuracy_score(y_test, y_pred)

# Top-3: is the correct answer in the model's top 3 predictions?
y_proba = model.predict_proba(X_test)
classes = model.classes_

# For each test row, get indices of the top 3 highest-probability classes
top3_indices = np.argsort(y_proba, axis=1)[:, -3:]
top3_predictions = classes[top3_indices]

# Check if the true label is anywhere in the top 3
top3_correct = np.array([
    y_test.iloc[i] in top3_predictions[i]
    for i in range(len(y_test))
])
top3_acc = top3_correct.mean()

print(f"Top-1 accuracy: {top1_acc * 100:.1f}%")
print(f"Top-3 accuracy: {top3_acc * 100:.1f}%")

Top-1 accuracy: 39.8%
Top-3 accuracy: 59.7%


V-2


In [16]:
import numpy as np

# Build features with cyclical month
features_v2 = df_clean[['latitude', 'longitude', 'year']].copy()
features_v2['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
features_v2['month_cos'] = np.cos(2 * np.pi * df_clean['month'] / 12)

target = df_clean['lower_level_incident_type']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    features_v2, target,
    test_size=0.2,
    random_state=42,
    stratify=target,
)

print(f"Train: {len(X_train2):,} rows, {X_train2.shape[1]} features")
print(f"Test:  {len(X_test2):,} rows")
print(f"Features: {list(features_v2.columns)}")

Train: 1,353,642 rows, 5 features
Test:  338,411 rows
Features: ['latitude', 'longitude', 'year', 'month_sin', 'month_cos']


In [17]:
import time

print("Training v2 with cyclical month...")
t = time.time()

model_v2 = HistGradientBoostingClassifier(
    max_iter=100,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=1,
)
model_v2.fit(X_train2, y_train2)

print(f"\nTrained in {(time.time() - t) / 60:.1f} minutes")

Training v2 with cyclical month...
Binning 0.049 GB of training data: 0.240 s
Binning 0.005 GB of validation data: 0.005 s
Fitting gradient boosted rounds:
Fit 1120 trees in 71.106 s, (34683 total leaves)
Time spent computing histograms: 21.767s
Time spent finding best splits:  1.140s
Time spent applying splits:      11.433s
Time spent predicting:           6.956s

Trained in 1.2 minutes


In [18]:
y_pred_v2 = model_v2.predict(X_test2)
top1_v2 = accuracy_score(y_test2, y_pred_v2)

y_proba_v2 = model_v2.predict_proba(X_test2)
top3_indices_v2 = np.argsort(y_proba_v2, axis=1)[:, -3:]
top3_predictions_v2 = model_v2.classes_[top3_indices_v2]
top3_correct_v2 = np.array([
    y_test2.iloc[i] in top3_predictions_v2[i]
    for i in range(len(y_test2))
])
top3_v2 = top3_correct_v2.mean()

print(f"Top-1 v2: {top1_v2 * 100:.1f}%  (v1 was 39.8%)")
print(f"Top-3 v2: {top3_v2 * 100:.1f}%  (v1 was 59.7%)")

Top-1 v2: 37.6%  (v1 was 39.8%)
Top-3 v2: 59.2%  (v1 was 59.7%)


In [19]:
import joblib
joblib.dump(model, 'predictor_v1.joblib')
joblib.dump(model_v2, 'predictor_v2.joblib')
joblib.dump(list(model.classes_), 'predictor_classes.joblib')
print("Saved v1 and v2.")

Saved v1 and v2.


In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
import time

# ---- Distance helper ----
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points in degrees."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# ---- Anchor cities (approximate centers) ----
CITIES = {
    'vilnius':   (54.687, 25.279),
    'kaunas':    (54.898, 23.903),
    'klaipeda':  (55.708, 21.131),
    'siauliai':  (55.934, 23.314),
    'panevezys': (55.733, 24.357),
}

# ---- Build features ----
print("Building features...")
features_v3 = pd.DataFrame()
features_v3['latitude']  = df_clean['latitude'].values
features_v3['longitude'] = df_clean['longitude'].values
features_v3['year']      = df_clean['year'].values
features_v3['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12).values
features_v3['month_cos'] = np.cos(2 * np.pi * df_clean['month'] / 12).values

# 5x5 grid covering Lithuania (approx bounds: lat 53.9-56.4, lon 20.9-26.9)
features_v3['grid_lat'] = pd.cut(df_clean['latitude'],  bins=5, labels=False).astype(int).values
features_v3['grid_lon'] = pd.cut(df_clean['longitude'], bins=5, labels=False).astype(int).values

# Distance to each city
for name, (city_lat, city_lon) in CITIES.items():
    features_v3[f'dist_{name}'] = haversine_km(
        df_clean['latitude'].values, df_clean['longitude'].values,
        city_lat, city_lon
    )

target = df_clean['lower_level_incident_type']

print(f"Features: {list(features_v3.columns)}")
print(f"Shape: {features_v3.shape}")

# ---- Train/test split ----
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    features_v3, target,
    test_size=0.2,
    random_state=42,
    stratify=target,
)

# ---- Compute sample weights for balanced training ----
print("\nComputing sample weights for class balance...")
sample_weights = compute_sample_weight('balanced', y_train3)
print(f"Sample weight range: {sample_weights.min():.4f} to {sample_weights.max():.4f}")

# ---- Train ----
print("\nTraining v3 with all improvements...")
t = time.time()
model_v3 = HistGradientBoostingClassifier(
    max_iter=100,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=1,
)
model_v3.fit(X_train3, y_train3, sample_weight=sample_weights)
print(f"\nTrained in {(time.time() - t) / 60:.1f} minutes")

# ---- Evaluate ----
print("\nEvaluating...")
y_pred_v3 = model_v3.predict(X_test3)
top1_v3 = accuracy_score(y_test3, y_pred_v3)

y_proba_v3 = model_v3.predict_proba(X_test3)
top3_indices_v3 = np.argsort(y_proba_v3, axis=1)[:, -3:]
top3_predictions_v3 = model_v3.classes_[top3_indices_v3]
top3_correct_v3 = np.array([
    y_test3.iloc[i] in top3_predictions_v3[i]
    for i in range(len(y_test3))
])
top3_v3 = top3_correct_v3.mean()

print(f"\nTop-1 v3: {top1_v3 * 100:.1f}%  (v1 was 39.8%, v2 was 37.6%)")
print(f"Top-3 v3: {top3_v3 * 100:.1f}%  (v1 was 59.7%, v2 was 59.2%)")

Building features...
Features: ['latitude', 'longitude', 'year', 'month_sin', 'month_cos', 'grid_lat', 'grid_lon', 'dist_vilnius', 'dist_kaunas', 'dist_klaipeda', 'dist_siauliai', 'dist_panevezys']
Shape: (1692053, 12)

Computing sample weights for class balance...
Sample weight range: 0.0217 to 1342.8988

Training v3 with all improvements...
Binning 0.117 GB of training data: 0.581 s
Binning 0.013 GB of validation data: 0.017 s
Fitting gradient boosted rounds:
Fit 1456 trees in 135.415 s, (45136 total leaves)
Time spent computing histograms: 52.214s
Time spent finding best splits:  2.305s
Time spent applying splits:      19.331s
Time spent predicting:           11.776s

Trained in 2.3 minutes

Evaluating...

Top-1 v3: 2.5%  (v1 was 39.8%, v2 was 37.6%)
Top-3 v3: 9.6%  (v1 was 59.7%, v2 was 59.2%)


In [21]:
print("Training v4: geography features WITHOUT class weighting...")
t = time.time()
model_v4 = HistGradientBoostingClassifier(
    max_iter=100,
    max_depth=8,
    learning_rate=0.1,
    random_state=42,
    verbose=1,
)
# Note: NO sample_weight argument this time
model_v4.fit(X_train3, y_train3)
print(f"\nTrained in {(time.time() - t) / 60:.1f} minutes")

y_pred_v4 = model_v4.predict(X_test3)
top1_v4 = accuracy_score(y_test3, y_pred_v4)

y_proba_v4 = model_v4.predict_proba(X_test3)
top3_indices_v4 = np.argsort(y_proba_v4, axis=1)[:, -3:]
top3_predictions_v4 = model_v4.classes_[top3_indices_v4]
top3_correct_v4 = np.array([
    y_test3.iloc[i] in top3_predictions_v4[i]
    for i in range(len(y_test3))
])
top3_v4 = top3_correct_v4.mean()

print(f"\nTop-1 v4: {top1_v4 * 100:.1f}%  (v1: 39.8%, v3: 2.5%)")
print(f"Top-3 v4: {top3_v4 * 100:.1f}%  (v1: 59.7%, v3: 9.6%)")

Training v4: geography features WITHOUT class weighting...
Binning 0.117 GB of training data: 0.444 s
Binning 0.013 GB of validation data: 0.014 s
Fitting gradient boosted rounds:
Fit 1120 trees in 76.079 s, (34686 total leaves)
Time spent computing histograms: 28.690s
Time spent finding best splits:  1.059s
Time spent applying splits:      11.242s
Time spent predicting:           5.589s

Trained in 1.3 minutes

Top-1 v4: 39.3%  (v1: 39.8%, v3: 2.5%)
Top-3 v4: 59.2%  (v1: 59.7%, v3: 9.6%)


In [23]:
import joblib

# v1 wins — save it as the canonical predictor
joblib.dump(model, 'predictor.joblib')
joblib.dump(list(model.classes_), 'predictor_classes.joblib')

# Also keep v3 and v4 around for reference (showing the experiments)
joblib.dump(model_v2, 'predictor_v2_cyclical.joblib')
joblib.dump(model_v3, 'predictor_v3_balanced.joblib')
joblib.dump(model_v4, 'predictor_v4_geography.joblib')

print("Saved predictor.joblib (the production v1) plus experimental variants.")

Saved predictor.joblib (the production v1) plus experimental variants.


In [ ]:
from sklearn.inspection import permutation_importance

print("Computing permutation importance (this takes a minute)...")
# Use a subset of test data to keep it fast
sample_size = 10000
X_sample = X_test.sample(sample_size, random_state=42)
y_sample = y_test.loc[X_sample.index]

result = permutation_importance(
    model, X_sample, y_sample,
    n_repeats=3,
    random_state=42,
    n_jobs=-1,
)

print("\nPermutation importances (higher = more important):")
for name, mean_imp, std_imp in zip(
    ['lat', 'lon', 'month', 'year'],
    result.importances_mean,
    result.importances_std,
):
    print(f"  {name}: {mean_imp:.4f} ± {std_imp:.4f}")

Computing permutation importance (this takes a minute)...

Permutation importances (higher = more important):
  lat: 0.0815 ± 0.0015
  lon: 0.0734 ± 0.0021
  month: 0.0010 ± 0.0009
  year: 0.0016 ± 0.0003


: 